# GPU pipeline: emotion extraction and valence–arousal ratings

This notebook is the **GPU** workflow for this project. A local CPU machine is not enough to load Qwen and score labels, so these steps are run on Google Colab (or another GPU).

**What this notebook does**
1. Extract **single-label** examples from [GoEmotions](https://huggingface.co/datasets/google-research-datasets/go_emotions) (all splits merged).
2. Query **Qwen/Qwen3-1.7B** for model-native **valence** and **arousal** ratings of all 28 emotion labels (27 GoEmotions emotions + `neutral`).

Live Colab copy: [open this notebook on Colab](https://colab.research.google.com/drive/1TEOHKThV3pn3VaTYYUxC3al-CW18DEkf)

**Runtime:** `Runtime → Change runtime type → GPU` (T4 is enough).

## 1. Check GPU

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("No GPU detected. In Colab: Runtime → Change runtime type → GPU.")

## 2. Install dependencies and load the repository

If you opened this file from GitHub, clone the repo. If you uploaded the project folder to Colab, skip the clone and `cd` into it instead.

In [ ]:
!pip install -q "transformers>=4.51" accelerate datasets huggingface_hub

# Public clone will fail while the GitHub repo is private. Use a PAT, or upload the project zip.
!git clone https://github.com/Shreya07099/valence-arousal-representation-in-llms.git
%cd valence-arousal-representation-in-llms

## 3. Extract single-label GoEmotions examples

GoEmotions is multi-label. This step keeps comments with **exactly one** emotion and writes one JSONL file (train + validation + test combined).

In [ ]:
!python scripts/pull_go_emotions.py --output data/processed/go_emotions_single_label.jsonl
!python scripts/count_emotion_labels.py --input data/processed/go_emotions_single_label.jsonl

## 4. Query Qwen for valence–arousal coordinates

Loads `Qwen/Qwen3-1.7B`, scores each of the 28 labels with three prompt templates, averages the parsed JSON, and clamps values to `[-1, +1]`.

This is the step that requires a GPU and is not practical on a local CPU.

In [ ]:
!python scripts/query_coordinates.py \
  --model Qwen/Qwen3-1.7B \
  --output outputs/qwen_native_coordinates.json

## 5. Inspect and save results

Optionally mount Drive so the JSON survives the Colab session.

In [ ]:
import json
from pathlib import Path

path = Path("outputs/qwen_native_coordinates.json")
coords = json.loads(path.read_text(encoding="utf-8"))
print(f"Labels: {len(coords)}")
for label, va in sorted(coords.items()):
    print(f"  {label:16s}  valence={va['valence']:+.2f}  arousal={va['arousal']:+.2f}")

In [ ]:
from google.colab import drive, files

drive.mount("/content/drive")
!cp outputs/qwen_native_coordinates.json /content/drive/MyDrive/qwen_native_coordinates.json
!cp data/processed/go_emotions_single_label.jsonl /content/drive/MyDrive/go_emotions_single_label.jsonl

files.download("outputs/qwen_native_coordinates.json")